In [1]:
import pandas as pd
import numpy as np
import scipy
import itertools
import copy
from scipy import signal
import matplotlib.pyplot as plt

In [2]:
np.random.seed(23)

## Generate data

In [3]:
# Causal graph
def generate_graph():
    graph = np.zeros((d,d))
    for i in range(d):
        for j in range(i):
            b = scipy.stats.bernoulli.rvs(p)
            if b == 1:
                graph[i,j] = 1
    return graph

In [4]:
# Mixing matrix
def generate_mixing():
    H = 1.0 + np.random.normal(size=(d,d))
    return H

In [5]:
# generate linear causal models z = (I-A)^{-1}eps, x = H^{-1}z
def generate_model(H,graph):
    res = []
    for k in range(K):
        A = np.zeros((d,d))
        for i in range(d):
            for j in range(i):
                if graph[i,j]==1:
                    A[i,j] = np.random.normal()
        nscale = np.absolute(np.random.normal(size=d))+1
        # nscale = np.random.uniform(1,2,size=d)
        res.append(np.diagflat(1/nscale)@(np.identity(d)-A)@H)
    return np.array(res) 

In [6]:
# generate noise variables from generalized normal distribution
def generate_noise(beta):
    noise = []
    for i in range(d):
        x = scipy.stats.gennorm.rvs(beta[i])
        var = scipy.stats.gennorm.var(beta[i])
        noise.append(x/(np.sqrt(var)))
    return np.asarray(noise)

In [7]:
def generate_signal(models,beta):
    signalList = []
    for k in range(K):
        new_signal = []
        for i in range(N):
            noise = generate_noise(beta)
            new_signal.append(np.matmul(np.linalg.inv(models[k]),noise))
        signalList.append(new_signal)
    return signalList

## Step 1. Linear ICA

In [8]:
from sklearn.decomposition import FastICA

In [9]:
# Distance of two matrices up to permutations
def angle_sin(u,v):
    return 1- (np.dot(u,v)/np.linalg.norm(u)/np.linalg.norm(v))**2
def dist(A,B):
    s = 0.0
    for i in range(d):
        new_s = 1
        for j in range(d):
            new_s = min(new_s, angle_sin(A[i],B[j]))
        s += new_s
    return s/d

In [10]:
# Get permutation of a matrix
def get_permute(M0, samples0, source=0):
    M = np.copy(M0)
    z0 = samples0 @ np.linalg.inv(M).T
    # each row is a sample of latent variables
    cnt = []
    for i in range(d):
        cnt.append((np.absolute(z0[:,i])<1).sum())

    for i in range(d):
        for j in range(d-i-1):
            if cnt[j] > cnt[j+1]:
                a = cnt[j]
                cnt[j] = cnt[j+1]
                cnt[j+1] = a
                for i in range(d):
                    a = M[i,j]
                    M[i,j] = M[i,j+1]
                    M[i,j+1] = a
    for i in range(d):
        if M[source,i]<0:
            M[:,i] = - M[:,i]
    return M

## Algorithm 1

In [11]:
# Some useful functions
def projection(v, W):
    return v - np.matmul(W.T@np.linalg.inv(W@W.T)@W, v)
def proj(v, W):
    return np.matmul(W.T@np.linalg.inv(W@W.T)@W, v)
def intersect(W):
    m = len(W)
    M = np.zeros((d,d), dtype=float)
    for i in range(m):
        P = np.identity(d)-W[i].T@np.linalg.inv(W[i]@W[i].T)@W[i]
        M += P.T@P
    U_, S_, Vh_ = np.linalg.svd(M)
    return Vh_[-1]

## Algorithm 2

In [12]:
def identify_parents(Matrices, S, i, tl):
    P, r = [], []
    lastS_ = np.ones(d,dtype=float)
    S_List = []
    for j in range(len(S)+1):
        rows = []
        W = Matrices[0,S[0:j],:]
        for k in range(K):
            
            rows.append(projection(Matrices[k,i],W))
        rows = np.array(rows)
        U_, S_, Vh_ = np.linalg.svd(rows)
        S_List.append(S_)
        if j == len(S):
            r.append(1)
        else:
            flag = 0
            for l in range(len(S_)):
                if S_[l] < tl:
                    r.append(l)
                    flag = 1
                    break
            if flag == 0:
                r.append(len(S_))
            lastS_ = np.copy(S_)
        if len(r) <= j:
            r.append(1)
        if j >= 1 and r[j] < r[j-1]:
            P.append(S[j-1])
    return P, S_List

## Algorithm 3

In [13]:
def learn_causal_model(Matrices, tl):
    S = []
    S_List = []
    E = np.zeros((d,d),dtype=int)
    Ch = [[i] for i in range(d)]
    deg = np.zeros(d)
    H = np.zeros((d,d), dtype=float)
    marked = np.zeros(d)
    while len(S) < d:
        gen = (i for i in range(d) if marked[i]==0)
        newidx = -1
        ratio = 2.0
        for i in gen:
            rows = []
            for k in range(K):
                if len(S)==0:
                    rows.append(Matrices[k,i])
                else:
                    W = Matrices[k,S]
                    rows.append(projection(Matrices[k,i],W))
            rows = np.array(rows)
            U_, S_, Vh_ = np.linalg.svd(rows)
            if S_[1]/S_[0] < ratio:
                ratio = S_[1]/S_[0]
                newidx = i
        i = newidx
        P, newS_List = identify_parents(Matrices, S, i, tl)
        S_List.append(newS_List)
        deg[i] = len(P)
        S.append(i)
        marked[i] = 1
        for j in P:
            E[i,j] = 1
            Ch[j].append(i)
    row_spaces = []
    for i in range(d):
        U_, S_, Vh_ = np.linalg.svd(Matrices[:,i,:])
        row_spaces.append(Vh_[0:int(deg[i])+1])
    for i in range(d):
        W = [row_spaces[j] for j in Ch[i]]
        h = intersect(W)
        H[i] = np.copy(h)
    return E, H, S_List, S

## Evaluation of the learning outcome

In [14]:
def matrix_sort_columns(H, source = 0):
    for i in range(d):
        for j in range(d-i-1):
            if abs(H[source,j])>abs(H[source,j+1]):
                for l in range(d):
                    a = H[l,j]
                    H[l,j] = H[l,j+1]
                    H[l,j+1] = a
    for i in range(d):
        if H[source,i] < 0:
            H[:,i] = -H[:,i]
    return H

In [15]:
def graph_isomorphism(G1, G2, pi):
    flag = 0
    for i in range(d):
        for j in range(d):
            if G1[i,j] != G2[pi[i],pi[j]]:
                flag = 1
    if flag == 0:
        return True
    return False

In [16]:
def mixing_error(H, Hres, G, source = 0):
    for i in range(d):
        Hres[i] /= np.linalg.norm(Hres[i])
    errList = np.ones(d,dtype=float)
    origErrList = np.ones(d,dtype=float)
    minerr = 100.0
    domList = []
    for i in range(d):
        newdomList = []
        for j in range(d):
            if G[i,j] != 0:
                flag = 0
                for l in range(d):
                    if G[l,i] != 0 and G[l,j] == 0:
                        flag = 1
                        break
                if flag == 0:
                    newdomList.append(j)
        newdomList.append(i)
        newdomList = np.array(newdomList)
        domList.append(newdomList)
        
    newHres = np.zeros((d,d),dtype=float)
    minpi = 0
    cnt = 0
    for pi in itertools.permutations(list(range(d))):
        
        newerrList = []
        for j in range(d):
            newHres[j] = np.copy(Hres[pi[j]])
        for i in range(d):
            newerrList.append(np.linalg.norm(projection(newHres[i],H[domList[i]])))
        newerrList = np.array(newerrList)
        newerr = np.linalg.norm(newerrList)
        if newerr < minerr:
            minerr = newerr
            minpi = copy.copy(pi)
            errList = np.copy(newerrList)
            origErrList = [np.linalg.norm(projection(newHres[i],H[[i]])) for i in range(d)]
        cnt += 1

    return errList, minpi, origErrList

In [17]:
def count_parents(pi, G, i, j):
    # number of parents of pi[i] in pi[j...i] including i
    cnt = 0
    for l in range(j,i+1):
        if G[pi[i],pi[l]] != 0:
            cnt +=1
    return cnt

In [18]:
# samples from different environments
beta = np.array([0.2*(i+1)**2 for i in range(10)])

## Running the experiment

In [43]:
def evaluation(d, K, N = 40000, p = 0.5, A = 10, tlList = [0.1, 0.15, 0.2, 0.25, 0.3]):
    graphList, HList, modelList, samplesList = [], [], [], []
    for i in range(A):
        G, H = generate_graph(), generate_mixing()
        graphList.append(G)
        HList.append(H)
        modelList.append(np.array(generate_model(H,G)))
        sampleList = generate_signal(modelList[i],beta)
        samplesList.append(sampleList)

    sampleSizeList = [(N//10) * i for i in range(10, 0, -1)]
    
    avgErrList = []
    recoveryCount = np.zeros((len(tlList), len(sampleSizeList)))
    
    for s in range(len(tlList)):
        tl = tlList[s]
        tl_avgErrList = []
    
        for i in range(A):
            tl_graph_avgErrList = []
            
            G = graphList[i]
            H = HList[i]
            models = modelList[i]
            sampleList = samplesList[i]
            
            ica = FastICA(n_components=d)
            flag = 0
            firstN = N
            
            for z in range(len(sampleSizeList)):
                M = sampleSizeList[z]
                
                mList = []
                for i in range(K):
                    S_ = ica.fit_transform(np.array(sampleList[i][0:M]))
                    A_ = ica.mixing_
                    A_ = get_permute(A_, sampleList[i])
                    mList.append(np.linalg.inv(A_))
                mList = np.array(mList)
                
                
                minEres, minHres, S_List, S = learn_causal_model(mList, tl)
                minerr, minpi, minOrig = mixing_error(H, minHres, G)
                
                
                newminEres = np.zeros((d,d),dtype=int)
                for x in range(d):
                    for y in range(d):
                        newminEres[x,y] = minEres[minpi[x],minpi[y]]
    
                tl_graph_avgErrList.append(sum(minerr)/d)
                
                if np.array_equal(G.astype(int),newminEres):
                    recoveryCount[s,z] += 1
                    
            
            tl_avgErrList.append(tl_graph_avgErrList)
        
        avgErrList.append(tl_avgErrList)

    opt_tl = np.argmax(recoveryCount[:,0])
    return np.array(avgErrList[opt_tl]), recoveryCount[opt_tl]

In [44]:
np.random.seed(23)
d = K = 5
avgErrList, recoveryCount = evaluation(d, K)

In [47]:
def plot(avgErrList, recoveryCount, N = 40000):
    fig, ax = plt.subplots(figsize=(12,6))
    invAvgErrList = avgErrList[:, ::-1]
    
    flierprops = dict(marker='o', markerfacecolor='white', markersize=5,
                      linestyle='none', markeredgecolor='maroon')
    box = ax.boxplot(invAvgErrList, 0, flierprops = flierprops, patch_artist=True, labels = [2*i for i in range(1,11)],
                     medianprops = dict(color="purple",linewidth=1.5), boxprops=dict(facecolor='mediumpurple', color='purple', alpha=0.3))
    
    ax.set_xlabel(r'Sample size ($\times$ ' + str(N//10) + ')', fontsize=18)
    ax.set_ylabel('SNA error', fontsize=18, color = 'purple')
    ax.set_yscale('log')
    
    inds = np.arange(1,11)
    ax2 = ax.twinx()
    ax2.set_ylabel('Graph recovery acc.', fontsize = 18, color = 'darkslategrey')
    
    ax2.plot(inds, [recoveryCount[i] for i in range(9,-1,-1)], marker = '*', markersize = 15, color = 'darkslategrey')

    return fig

In [ ]:
fig = plot(avgErrList, recoveryCount)

fig.tight_layout()
# fig.savefig('boxplot.png', dpi=300)